In [ ]:
from datetime import datetime, timedelta, timezone
import pandas as pd
import numpy as np
from cognite.client import CogniteClient
from cognite.client.data_classes import EventWrite
from cognite.client.exceptions import CogniteNotFoundError

# Initialize Cognite Client
client = CogniteClient()

# Set local operational timezone: GMT-4
LOCAL_TZ = timezone(timedelta(hours=-4))

# D&I Process Constants
CANS_PER_SHORT_CAN = 5           # Cans lost per short can defect
CANS_PER_TRIMMER_JAM = 9        # Cans lost per trimmer jam
DOWNTIME_PER_SHORT_CAN_MIN = 5.0 # Minutes stopped per short can
DOWNTIME_PER_TRIM_JAM_MIN = 3.0 # Minutes stopped per trimmer jam
CAN_WEIGHT_KG = 0.0093           # Weight per aluminum can (~9.3g)

# Unified Machine Configurations
MACHINE_CONFIGS = [
    # --- PRINTERS ---
    {
        "code": "p11",
        "machine_type": "printer",
        "asset_ext_id": "ActivoSimulacion.L1.PRINTER",
        "nominal_capacity": 66000.0,
        "ts_prod": "ActivoSimulacion.L1.PRINTER.PRINTER_L1_PROD_ACT",
        "ts_retract": "ActivoSimulacion.L1.PRINTER.PRINTER_L1_RETRACT_ACT",
        "ts_blow_off": "ActivoSimulacion.L1.PRINTER.PRINTER_L1_LAT_SOP_ACT",
    },
    {
        "code": "p31",
        "machine_type": "printer",
        "asset_ext_id": "ActivoSimulacion.L3.PRINTER.PRINTER31",
        "nominal_capacity": 66000.0,
        "ts_prod": "ActivoSimulacion.L3.PRINTER.PRINTER31.PRINTER_L31_PROD_ACT",
        "ts_retract": "ActivoSimulacion.L3.PRINTER.PRINTER31.PRINTER_L31_RETRACT_ACT",
        "ts_blow_off": "ActivoSimulacion.L3.PRINTER.PRINTER31.PRINTER_L31_LAT_SOP_ACT",
    },
    {
        "code": "p32",
        "machine_type": "printer",
        "asset_ext_id": "ActivoSimulacion.L3.PRINTER.PRINTER32",
        "nominal_capacity": 66000.0,
        "ts_prod": "ActivoSimulacion.L3.PRINTER.PRINTER32.PRINTER_L32_PROD_ACT",
        "ts_retract": "ActivoSimulacion.L3.PRINTER.PRINTER32.PRINTER_L32_RETRACT_ACT",
        "ts_blow_off": "ActivoSimulacion.L3.PRINTER.PRINTER32.PRINTER_L32_LAT_SOP_ACT",
    },
    # --- D&I MACHINERY ---
    {
        "code": "di11",
        "machine_type": "di",
        "asset_ext_id": "ActivoSimulacion.L1.DI.DI11",
        "ts_prod": "ActivoSimulacion.L1.DI.DI11.DI11_PROD_ACT",
        "ts_short_cans": "ActivoSimulacion.L1.DI.DI11.DI11_LATAS_CORTAS_ACT",
        "ts_trimmer_jams": "ActivoSimulacion.L1.DI.DI11.DI11_TRANC_TRIM_ACT",
    },
    {
        "code": "di12",
        "machine_type": "di",
        "asset_ext_id": "ActivoSimulacion.L1.DI.DI12",
        "ts_prod": "ActivoSimulacion.L1.DI.DI12.DI12_PROD_ACT",
        "ts_short_cans": "ActivoSimulacion.L1.DI.DI12.DI12_LATAS_CORTAS_ACT",
        "ts_trimmer_jams": "ActivoSimulacion.L1.DI.DI12.DI12_TRANC_TRIM_ACT",
    },
    {
        "code": "di14",
        "machine_type": "di",
        "asset_ext_id": "ActivoSimulacion.L1.DI.DI14",
        "ts_prod": "ActivoSimulacion.L1.DI.DI14.DI14_PROD_ACT",
        "ts_short_cans": "ActivoSimulacion.L1.DI.DI14.DI14_LATAS_CORTAS_ACT",
        "ts_trimmer_jams": "ActivoSimulacion.L1.DI.DI14.DI14_TRANC_TRIM_ACT",
    },
    {
        "code": "di15",
        "machine_type": "di",
        "asset_ext_id": "ActivoSimulacion.L1.DI.DI15",
        "ts_prod": "ActivoSimulacion.L1.DI.DI15.DI15_PROD_ACT",
        "ts_short_cans": "ActivoSimulacion.L1.DI.DI15.DI15_LATAS_CORTAS_ACT",
        "ts_trimmer_jams": "ActivoSimulacion.L1.DI.DI15.DI15_TRANC_TRIM_ACT",
    },
    {
        "code": "di17",
        "machine_type": "di",
        "asset_ext_id": "ActivoSimulacion.L1.DI.DI17",
        "ts_prod": "ActivoSimulacion.L1.DI.DI17.DI17_PROD_ACT",
        "ts_short_cans": "ActivoSimulacion.L1.DI.DI17.DI17_LATAS_CORTAS_ACT",
        "ts_trimmer_jams": "ActivoSimulacion.L1.DI.DI17.DI17_TRANC_TRIM_ACT",
    },
    {
        "code": "di18",
        "machine_type": "di",
        "asset_ext_id": "ActivoSimulacion.L1.DI.DI18",
        "ts_prod": "ActivoSimulacion.L1.DI.DI18.DI18_PROD_ACT",
        "ts_short_cans": "ActivoSimulacion.L1.DI.DI18.DI18_LATAS_CORTAS_ACT",
        "ts_trimmer_jams": "ActivoSimulacion.L1.DI.DI18.DI18_TRANC_TRIM_ACT",
    },
    # --- STANDUM MACHINERY ---
    {
        "code": "standum31",
        "machine_type": "standum",
        "asset_ext_id": "ActivoSimulacion.L3.STANDUM.STANDUM31",
        "ts_prod": "ActivoSimulacion.L3.STANDUM.STANDUM31.STANDUM31_PROD_ACT",
        "ts_short_cans": "ActivoSimulacion.L3.STANDUM.STANDUM31.STANDUM31_PROD_LATAS_CORTAS_ACT",
        "ts_trimmer_jams": "ActivoSimulacion.L3.STANDUM.STANDUM31.STANDUM31_TRANC_TRIM_ACT",
    },
    {
        "code": "standum32",
        "machine_type": "standum",
        "asset_ext_id": "ActivoSimulacion.L3.STANDUM.STANDUM32",
        "ts_prod": "ActivoSimulacion.L3.STANDUM.STANDUM32.STANDUM32_PROD_ACT",
        "ts_short_cans": "ActivoSimulacion.L3.STANDUM.STANDUM32.STANDUM32_PROD_LATAS_CORTAS_ACT",
        "ts_trimmer_jams": "ActivoSimulacion.L3.STANDUM.STANDUM32.STANDUM32_TRANC_TRIM_ACT",
    },
    {
        "code": "standum33",
        "machine_type": "standum",
        "asset_ext_id": "ActivoSimulacion.L3.STANDUM.STANDUM33",
        "ts_prod": "ActivoSimulacion.L3.STANDUM.STANDUM33.STANDUM33_PROD_ACT",
        "ts_short_cans": "ActivoSimulacion.L3.STANDUM.STANDUM33.STANDUM33_PROD_LATAS_CORTAS_ACT",
        "ts_trimmer_jams": "ActivoSimulacion.L3.STANDUM.STANDUM33.STANDUM33_TRANC_TRIM_ACT",
    },
    {
        "code": "standum34",
        "machine_type": "standum",
        "asset_ext_id": "ActivoSimulacion.L3.STANDUM.STANDUM34",
        "ts_prod": "ActivoSimulacion.L3.STANDUM.STANDUM34.STANDUM34_PROD_ACT",
        "ts_short_cans": "ActivoSimulacion.L3.STANDUM.STANDUM34.STANDUM34_PROD_LATAS_CORTAS_ACT",
        "ts_trimmer_jams": "ActivoSimulacion.L3.STANDUM.STANDUM34.STANDUM34_TRANC_TRIM_ACT",
    },
    {
        "code": "standum35",
        "machine_type": "standum",
        "asset_ext_id": "ActivoSimulacion.L3.STANDUM.STANDUM35",
        "ts_prod": "ActivoSimulacion.L3.STANDUM.STANDUM35.STANDUM35_PROD_ACT",
        "ts_short_cans": "ActivoSimulacion.L3.STANDUM.STANDUM35.STANDUM35_PROD_LATAS_CORTAS_ACT",
        "ts_trimmer_jams": "ActivoSimulacion.L3.STANDUM.STANDUM35.STANDUM35_TRANC_TRIM_ACT",
    },
    {
        "code": "standum36",
        "machine_type": "standum",
        "asset_ext_id": "ActivoSimulacion.L3.STANDUM.STANDUM36",
        "ts_prod": "ActivoSimulacion.L3.STANDUM.STANDUM36.STANDUM36_PROD_ACT",
        "ts_short_cans": "ActivoSimulacion.L3.STANDUM.STANDUM36.STANDUM36_PROD_LATAS_CORTAS_ACT",
        "ts_trimmer_jams": "ActivoSimulacion.L3.STANDUM.STANDUM36.STANDUM36_TRANC_TRIM_ACT",
    },
    {
        "code": "standum37",
        "machine_type": "standum",
        "asset_ext_id": "ActivoSimulacion.L3.STANDUM.STANDUM37",
        "ts_prod": "ActivoSimulacion.L3.STANDUM.STANDUM37.STANDUM37_PROD_ACT",
        "ts_short_cans": "ActivoSimulacion.L3.STANDUM.STANDUM37.STANDUM37_PROD_LATAS_CORTAS_ACT",
        "ts_trimmer_jams": "ActivoSimulacion.L3.STANDUM.STANDUM37.STANDUM37_TRANC_TRIM_ACT",
    },
    {
        "code": "standum38",
        "machine_type": "standum",
        "asset_ext_id": "ActivoSimulacion.L3.STANDUM.STANDUM38",
        "ts_prod": "ActivoSimulacion.L3.STANDUM.STANDUM38.STANDUM38_PROD_ACT",
        "ts_short_cans": "ActivoSimulacion.L3.STANDUM.STANDUM38.STANDUM38_PROD_LATAS_CORTAS_ACT",
        "ts_trimmer_jams": "ActivoSimulacion.L3.STANDUM.STANDUM38.STANDUM38_TRANC_TRIM_ACT",
    },
]

HOUR_INTERVAL_MAP = {
    5: "5 a 6",   6: "6 a 7",   7: "7 a 8",   8: "8 a 9",
    9: "9 a 10", 10: "10 a 11", 11: "11 a 12", 12: "12 a 1",
    13: "1 a 2", 14: "2 a 3",  15: "3 a 4",  16: "4 a 5",
    17: "5 a 6", 18: "6 a 7",  19: "7 a 8",  20: "8 a 9",
    21: "9 a 10", 22: "10 a 11", 23: "11 a 12", 0: "12 a 1",
    1: "1 a 2",  2: "2 a 3",   3: "3 a 4",   4: "4 a 5"
}


In [ ]:
def get_asset_id(client: CogniteClient, identifier: str) -> int:
    asset = client.assets.retrieve(external_id=identifier)
    if asset:
        return asset.id

    res = client.assets.search(name=identifier, limit=5)
    if res:
        return res[0].id

    raise ValueError(f"Could not find asset '{identifier}' in CDF.")


def calculate_hourly_counter_delta(client: CogniteClient, external_id: str, start_ms: int, end_ms: int):
    """
    Calculates step-by-step counter accumulation and handles mid-hour resets.
    """
    try:
        dps = client.time_series.data.retrieve(
            external_id=external_id,
            start=start_ms,
            end=end_ms,
            limit=None,
            ignore_unknown_ids=True,
        )

        if not dps or len(dps) == 0:
            print(f"  [Warning] [{external_id}] No datapoints found in window.")
            return 0.0, False

        first_value = float(dps[0].value)
        last_value = float(dps[-1].value)

        if len(dps) == 1:
            print(f"  [{external_id}] First: {first_value:,.1f} | Last: {last_value:,.1f} | Delta: 0.0 | Reset: False")
            return 0.0, False

        hourly_delta = 0.0
        reset_occurred = False

        # Accumulate deltas step-by-step
        for i in range(1, len(dps)):
            prev_val = float(dps[i - 1].value)
            curr_val = float(dps[i].value)
            step_diff = curr_val - prev_val

            if step_diff < 0:
                reset_occurred = True
                hourly_delta += max(0.0, curr_val)
            else:
                hourly_delta += step_diff

        print(f"  [{external_id}] First: {first_value:,.1f} | Last: {last_value:,.1f} | Delta: {hourly_delta:,.1f} | Reset: {reset_occurred}")
        return hourly_delta, reset_occurred

    except CogniteNotFoundError:
        print(f"  [Warning] TimeSeries '{external_id}' not found.")
        return 0.0, False
    except Exception as e:
        print(f"  [Error] Reading '{external_id}': {e}")
        return 0.0, False


In [ ]:
def generate_printer_event(cfg: dict, start_ms: int, end_ms: int, last_hour_start_local: datetime, asset_id: int):
    printer_code = cfg["code"]
    nominal_cap = cfg["nominal_capacity"]

    start_hour_local = last_hour_start_local.hour
    shift_code = "day" if 5 <= start_hour_local < 17 else "night"
    date_str = last_hour_start_local.strftime("%Y%m%d")

    shift_start_hour = 5 if shift_code == "day" else 17
    hour_index = ((start_hour_local - shift_start_hour) % 24) + 1
    entry_slot = f"{hour_index:02d}"
    hour_interval = HOUR_INTERVAL_MAP.get(start_hour_local, f"{start_hour_local} a {start_hour_local + 1}")

    # Query CDF counters
    hourly_production, prod_reset = calculate_hourly_counter_delta(client, cfg["ts_prod"], start_ms, end_ms)
    hourly_retrac, retrac_reset = calculate_hourly_counter_delta(client, cfg["ts_retract"], start_ms, end_ms)
    blow_off, blowoff_reset = calculate_hourly_counter_delta(client, cfg["ts_blow_off"], start_ms, end_ms)

    # Compute KPI metrics
    downtime_minutes = max(0.0, round(60.0 - ((hourly_production * 60.0) / nominal_cap), 2))
    efficiency = round((hourly_production * 100.0) / nominal_cap, 2)

    event_ext_id = f"report_{printer_code}_{date_str}_{shift_code}_entry_{entry_slot}"

    resets = []
    if prod_reset:
        resets.append("prod")
    if retrac_reset:
        resets.append("retrac")
    if blowoff_reset:
        resets.append("blow_off")

    obs_text = f"Resets detected: {', '.join(resets)}" if resets else "Standard operation"

    report_event = EventWrite(
        external_id=event_ext_id,
        type="Production Report",
        subtype="Hourly Entry",
        start_time=start_ms,
        end_time=end_ms,
        description=f"Production Report {hour_interval} for Printer {printer_code.upper()}",
        asset_ids=[asset_id],
        metadata={
            "timezone": "GMT-4",
            "printer_code": printer_code,
            "shift": shift_code,
            "hour_interval": hour_interval,
            "hourly_production": str(hourly_production),
            "hourly_retrac": str(hourly_retrac),
            "blow_off": str(blow_off),
            "downtime_minutes": str(downtime_minutes),
            "efficiency": f"{efficiency:.2f}%",
            "observations": obs_text,
        },
    )

    res = client.events.upsert(report_event)
    print(f"  --> Successfully posted Printer Event: '{res.external_id}' (CDF ID: {res.id})")


def generate_di_event(cfg: dict, start_ms: int, end_ms: int, last_hour_start_local: datetime, asset_id: int):
    machine_code = cfg["code"]

    start_hour_local = last_hour_start_local.hour
    shift_code = "day" if 5 <= start_hour_local < 17 else "night"
    date_str = last_hour_start_local.strftime("%Y%m%d")

    shift_start_hour = 5 if shift_code == "day" else 17
    hour_index = ((start_hour_local - shift_start_hour) % 24) + 1
    entry_slot = f"{hour_index:02d}"
    hour_interval = HOUR_INTERVAL_MAP.get(start_hour_local, f"{start_hour_local} a {start_hour_local + 1}")

    # Query CDF counters
    prod_count, prod_reset = calculate_hourly_counter_delta(client, cfg["ts_prod"], start_ms, end_ms)
    short_count, short_reset = calculate_hourly_counter_delta(client, cfg["ts_short_cans"], start_ms, end_ms)
    trim_count, trim_reset = calculate_hourly_counter_delta(client, cfg["ts_trimmer_jams"], start_ms, end_ms)

    # Compute D&I Metrics
    cans_from_short = short_count * CANS_PER_SHORT_CAN
    cans_from_trim = trim_count * CANS_PER_TRIMMER_JAM

    downtime_short = short_count * DOWNTIME_PER_SHORT_CAN_MIN
    downtime_trim = trim_count * DOWNTIME_PER_TRIM_JAM_MIN
    total_downtime_min = min(60.0, downtime_short + downtime_trim)

    total_scrap_cans = cans_from_short + cans_from_trim
    merma_kg = round(total_scrap_cans * CAN_WEIGHT_KG, 2)

    total_produced_and_lost = prod_count + total_scrap_cans
    pct_merma = round((total_scrap_cans / total_produced_and_lost * 100.0), 2) if total_produced_and_lost > 0 else 0.0
    pct_eficiencia = round(((60.0 - total_downtime_min) / 60.0 * 100.0), 2)

    # Observations text
    resets = []
    if prod_reset:
        resets.append("prod")
    if short_reset:
        resets.append("short_cans")
    if trim_reset:
        resets.append("trimmer_jams")

    obs_parts = []
    if total_downtime_min == 0:
        obs_parts.append("Operación normal")
    elif short_count > 0 and trim_count > 0:
        obs_parts.append("Parada por latas cortas y trancamiento")
    elif short_count > 0:
        obs_parts.append("Parada por latas cortas")
    else:
        obs_parts.append("Parada por trancamiento trimmer")

    if resets:
        obs_parts.append(f"(Resets: {', '.join(resets)})")

    obs_text = " ".join(obs_parts)

    event_ext_id = f"report_{machine_code}_{date_str}_{shift_code}_entry_{entry_slot}"

    report_event = EventWrite(
        external_id=event_ext_id,
        type="Production Report",
        subtype="Hourly Entry DI",
        start_time=start_ms,
        end_time=end_ms,
        description=f"Production Report {hour_interval} for D&I Machine {machine_code.upper()}",
        asset_ids=[asset_id],
        metadata={
            "timezone": "GMT-4",
            "machine_code": machine_code.upper(),
            "shift": shift_code,
            "hour_interval": hour_interval,
            "hourly_production": str(int(prod_count)),
            "short_cans_per_hour": str(int(short_count)),
            "trimmer_jams_per_hour": str(int(trim_count)),
            "cans_by_short_can": str(int(cans_from_short)),
            "cans_by_trimmer_jam": str(int(cans_from_trim)),
            "downtime_by_short_can_min": f"{downtime_short:.2f}",
            "downtime_by_trimmer_jam_min": f"{downtime_trim:.2f}",
            "total_downtime_min": f"{total_downtime_min:.2f}",
            "merma_kg": f"{merma_kg:.2f}",
            "pct_merma": f"{pct_merma:.2f}%",
            "pct_eficiencia": f"{pct_eficiencia:.2f}%",
            "observations": obs_text,
        },
    )

    res = client.events.upsert(report_event)
    print(f"  --> Successfully posted D&I Event: '{res.external_id}' (CDF ID: {res.id})")


def generate_standum_event(cfg: dict, start_ms: int, end_ms: int, last_hour_start_local: datetime, asset_id: int):
    machine_code = cfg["code"]

    start_hour_local = last_hour_start_local.hour
    shift_code = "day" if 5 <= start_hour_local < 17 else "night"
    date_str = last_hour_start_local.strftime("%Y%m%d")

    shift_start_hour = 5 if shift_code == "day" else 17
    hour_index = ((start_hour_local - shift_start_hour) % 24) + 1
    entry_slot = f"{hour_index:02d}"
    hour_interval = HOUR_INTERVAL_MAP.get(start_hour_local, f"{start_hour_local} a {start_hour_local + 1}")

    hourly_production, prod_reset = calculate_hourly_counter_delta(client, cfg["ts_prod"], start_ms, end_ms)
    short_count, short_reset = calculate_hourly_counter_delta(client, cfg["ts_short_cans"], start_ms, end_ms)
    trim_count, trim_reset = calculate_hourly_counter_delta(client, cfg["ts_trimmer_jams"], start_ms, end_ms)

    cans_from_short = short_count * CANS_PER_SHORT_CAN
    cans_from_trim = trim_count * CANS_PER_TRIMMER_JAM
    total_downtime_min = min(60.0, (short_count * DOWNTIME_PER_SHORT_CAN_MIN) + (trim_count * DOWNTIME_PER_TRIM_JAM_MIN))
    total_scrap_cans = cans_from_short + cans_from_trim
    merma_kg = round(total_scrap_cans * CAN_WEIGHT_KG, 2)

    resets = []
    if prod_reset:
        resets.append("prod")
    if short_reset:
        resets.append("short_cans")
    if trim_reset:
        resets.append("trimmer_jams")

    obs_parts = []
    if total_downtime_min == 0:
        obs_parts.append("Operación normal")
    elif short_count > 0 and trim_count > 0:
        obs_parts.append("Parada por latas cortas y trancamiento")
    elif short_count > 0:
        obs_parts.append("Parada por latas cortas")
    else:
        obs_parts.append("Parada por trancamiento trimmer")

    if resets:
        obs_parts.append(f"(Resets: {', '.join(resets)})")

    obs_text = " ".join(obs_parts)

    event_ext_id = f"report_{machine_code}_{date_str}_{shift_code}_entry_{entry_slot}"

    report_event = EventWrite(
        external_id=event_ext_id,
        type="Production Report",
        subtype="Hourly Entry Standum",
        start_time=start_ms,
        end_time=end_ms,
        description=f"Production Report {hour_interval} for Standum {machine_code.upper()}",
        asset_ids=[asset_id],
        metadata={
            "timezone": "GMT-4",
            "machine_code": machine_code.upper(),
            "shift": shift_code,
            "hour_interval": hour_interval,
            "hourly_production": str(int(hourly_production)),
            "short_cans_per_hour": str(int(short_count)),
            "trimmer_jams_per_hour": str(int(trim_count)),
            "cans_by_short_can": str(int(cans_from_short)),
            "cans_by_trimmer_jam": str(int(cans_from_trim)),
            "total_downtime_min": f"{total_downtime_min:.2f}",
            "merma_kg": f"{merma_kg:.2f}",
            "observations": obs_text,
        },
    )

    res = client.events.upsert(report_event)
    print(f"  --> Successfully posted Standum Event: '{res.external_id}' (CDF ID: {res.id})")


def run_all_production_reports():
    if "get_asset_id" not in globals():
        def get_asset_id(client: CogniteClient, identifier: str) -> int:
            asset = client.assets.retrieve(external_id=identifier)
            if asset:
                return asset.id

            res = client.assets.search(name=identifier, limit=5)
            if res:
                return res[0].id

            raise ValueError(f"Could not find asset '{identifier}' in CDF.")

    now_local = datetime.now(LOCAL_TZ)
    last_hour_end_local = now_local.replace(minute=0, second=0, microsecond=0)
    last_hour_start_local = last_hour_end_local - timedelta(hours=1)

    start_ms = int(last_hour_start_local.timestamp() * 1000)
    end_ms = int(last_hour_end_local.timestamp() * 1000)

    print("=" * 80)
    print(f"EXECUTION WINDOW (GMT-4): {last_hour_start_local.strftime('%Y-%m-%d %H:%M')} to {last_hour_end_local.strftime('%H:%M')}")
    print("=" * 80)

    for cfg in MACHINE_CONFIGS:
        try:
            m_code = cfg["code"].upper()
            m_type = cfg["machine_type"].upper()
            print(f"\n--- Processing [{m_type}]: {m_code} ({cfg['asset_ext_id']}) ---")

            asset_id = get_asset_id(client, cfg["asset_ext_id"])

            if cfg["machine_type"] == "printer":
                generate_printer_event(cfg, start_ms, end_ms, last_hour_start_local, asset_id)
            elif cfg["machine_type"] == "standum":
                generate_standum_event(cfg, start_ms, end_ms, last_hour_start_local, asset_id)
            elif cfg["machine_type"] == "di":
                generate_di_event(cfg, start_ms, end_ms, last_hour_start_local, asset_id)
            else:
                print(f"  [Warning] Unsupported machine type: {cfg['machine_type']}")

        except ValueError as err:
            print(f"  [Error] {err} Skipping {m_code}...")
            continue
        except Exception as err:
            print(f"  [Error] {err} Skipping {m_code}...")
            continue


In [4]:
run_all_production_reports()

EXECUTION WINDOW (GMT-4): 2026-08-01 14:00 to 15:00

--- Processing [PRINTER]: P11 (ActivoSimulacion.L1.PRINTER) ---
  [ActivoSimulacion.L1.PRINTER.PRINTER_L1_PROD_ACT] First: 37.0 | Last: 537.0 | Delta: 744.0 | Reset: True
  [ActivoSimulacion.L1.PRINTER.PRINTER_L1_RETRACT_ACT] First: 3.0 | Last: 22.0 | Delta: 32.0 | Reset: True
  [ActivoSimulacion.L1.PRINTER.PRINTER_L1_LAT_SOP_ACT] First: 1.0 | Last: 26.0 | Delta: 35.0 | Reset: True
  --> Successfully posted Printer Event: 'report_p11_20260801_day_entry_10' (CDF ID: 1525345901280379)

--- Processing [PRINTER]: P31 (ActivoSimulacion.L3.PRINTER.PRINTER31) ---
  [ActivoSimulacion.L3.PRINTER.PRINTER31.PRINTER_L31_PROD_ACT] First: 38.0 | Last: 516.0 | Delta: 713.0 | Reset: True
  [ActivoSimulacion.L3.PRINTER.PRINTER31.PRINTER_L31_RETRACT_ACT] First: 1.0 | Last: 21.0 | Delta: 30.0 | Reset: True
  [ActivoSimulacion.L3.PRINTER.PRINTER31.PRINTER_L31_LAT_SOP_ACT] First: 3.0 | Last: 22.0 | Delta: 34.0 | Reset: True
  --> Successfully posted Prin